<a href="https://colab.research.google.com/github/knight19720208ui/TIP_Taller_BigData/blob/main/Taller1_Retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql import SparkSession

# Configuración del motor Spark
spark = SparkSession.builder.appName("TIP_Telecom").config("spark.sql.shuffle.partitions", "8").getOrCreate()

print("Generando base de 800,000 suscriptores y 12 millones de registros CDR (Call Detail Records)...")

# Generación de datos adaptada al contexto de Telecomunicaciones
df_suscriptores = spark.range(800000).withColumnRenamed("id", "id_suscriptor") \
    .withColumn("tipo_plan", F.when(F.rand() < 0.6, "Prepago").otherwise("Postpago"))

df_cdr = spark.range(12000000).withColumnRenamed("id", "id_registro") \
    .withColumn("id_suscriptor", (F.rand(seed=10) * 800000).cast("int")) \
    .withColumn("numero_destino", F.concat(F.lit("3"), F.lpad(((F.rand(seed=11) * 900000000 + 100000000).cast("int")).cast("string"), 9, "0"))) \
    .withColumn("duracion_llamada_seg", (F.rand(seed=12) * 1200 + 1).cast("int"))

# ---------------------------------------------------------
# RETO DE NEGOCIO: Encontrar el Top 3 de números más llamados por cada suscriptor
# ---------------------------------------------------------
# PISTA 1: Es necesario cruzar la información (JOIN) entre df_cdr y df_suscriptores utilizando la clave "id_suscriptor".
# PISTA 2: Defina un objeto de tipo Window. La partición debe hacerse por "id_suscriptor".
#          El orden debe establecerse sobre la columna "duracion_llamada_seg" de forma descendente (desc).
# PISTA 3: Aplique la función F.row_number() sobre la ventana definida para asignar un puesto (1, 2, 3...).
#          Recuerde que row_number() no deja huecos en caso de empates, a diferencia de rank().
# PISTA 4: Filtre el DataFrame resultante manteniendo únicamente las filas donde el puesto asignado sea menor o igual a 3.

# --- ESCRIBA SU CÓDIGO AQUÍ ---



# --- ZONA DE PRUEBAS (No modificar) ---
try:
    # Si la variable resultante se llama df_top3_contactos, este bloque mostrará la validación
    print("Ejecutando validación del proceso distribuido...")
    tiempo_inicio = time.time()

    total_registros = df_top3_contactos.count()

    print(f"Total de registros procesados (Aprox. 2.4 millones): {total_registros:,}")
    print(f"Tiempo de ejecución total: {time.time() - tiempo_inicio:.2f} segundos")
    print("\nEjemplo de salida para el suscriptor 500:")
    df_top3_contactos.filter(F.col("id_suscriptor") == 500).select("id_suscriptor", "numero_destino", "duracion_llamada_seg").show()

except NameError:
    print("ERROR: Aún no ha creado la variable 'df_top3_contactos' o existe un error de sintaxis en el bloque superior.")
except Exception as e:
    print(f"ERROR en la lógica de procesamiento: {e}")